In [1]:
tabla='qtian10'
col_tabla = 'infopecrefecane'
dat= "dat_ceqx003_essi"
col_dat='fec_cre'
essi='essi_dat_cqx003_etl'
col_essi='fec_cre'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [6]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [7]:

fecha= '2023-05-01'

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

numdoc = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
numdoc["num_doc"]=numdoc["num_doc"].str.strip()
numdoc["num_doc"]=numdoc["num_doc"].str.replace(' ', '', regex=True)
numdoc=numdoc.drop_duplicates(subset="num_doc")

tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
tipdoc["cod_tdo"]=tipdoc["cod_tdo"].str.strip()

cqxdesegrsop=pd.read_sql_query(f"SELECT id_desegr,des_cod FROM dim_cqxdesegrsop", con=connection2)
cqxdesegrsop['des_cod']=cqxdesegrsop['des_cod'].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)


In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(41860, 29)

In [11]:
#Eliminamos las columnas que no se usarán aquí: las descripciones y otras 4 más, previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['rec_gas', 'inf_dre','des_des','des_cas', 'des_red']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [12]:
base1.shape

(41860, 24)

In [13]:
inicioProceso=time.time()

In [14]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [15]:
base1.shape

(41860, 24)

In [16]:
control_a.append(base1.shape[0])

In [17]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(41860, 25)

In [18]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [19]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(41860, 25)

In [20]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [21]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(41860, 25)

In [22]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [23]:
merge.shape

(41860, 25)

In [24]:
numdoc=numdoc.rename(columns={"num_doc": "usu_mod","id_person": "id_usumod"})
base1['usu_mod']=base1['usu_mod'].str.strip()
base1["usu_mod"]=base1["usu_mod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='usu_mod')
base1=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1.shape

(41860, 25)

In [25]:
merge.shape #Se reduce mucho si es inner

(16064, 25)

In [26]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [27]:
numdoc=numdoc.rename(columns={"usu_mod": "usu_cre","id_usumod": "id_usucre"})
base1['usu_cre']=base1['usu_cre'].str.strip()
base1["usu_cre"]=base1["usu_cre"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='usu_cre')
base1=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1.shape

(41860, 25)

In [28]:
merge.shape #Puede ser inner

(41860, 25)

In [29]:
log_falla('id_usucre', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [30]:
numdoc=numdoc.rename(columns={"usu_cre": "num_doc","id_usucre": "id_numdoc"})
base1['num_doc']=base1['num_doc'].str.strip()
base1["num_doc"]=base1["num_doc"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='num_doc')
base1=pd.merge(base1,numdoc,how='left',on='num_doc')
base1.shape

(41860, 25)

In [31]:
merge.shape #Puede ser inner

(41860, 25)

In [32]:
log_falla('id_numdoc', 'num_doc', True)
log_control('dim_personal')
base1=base1.drop('num_doc',axis=1)

In [33]:
tipdoc=tipdoc.rename(columns={"cod_tdo": "cod_tdi"})
base1['cod_tdi']=base1['cod_tdi'].str.strip()
base1["cod_tdi"]=base1["cod_tdi"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,tipdoc,how='inner',on='cod_tdi')
base1=pd.merge(base1,tipdoc,how='left',on='cod_tdi')
base1.shape

(41860, 25)

In [34]:
merge.shape #Puede ser inner

(41860, 25)

In [35]:
log_falla('id_tipdoc', 'cod_tdi', True)
log_control('dim_tipdoc')
base1=base1.drop('cod_tdi',axis=1)

In [36]:
pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(41860, 25)

In [37]:
merge.shape

(41253, 25)

In [38]:
log_falla('id_pacsec', 'pac_sec', True)
log_control('dim_pacsec')
base1=base1.drop('pac_sec',axis=1)

In [39]:
cqxdesegrsop=cqxdesegrsop.rename(columns={"des_cod": "cod_des","id_desegr": "id_desope"})
base1['cod_des']=base1['cod_des'].str.strip()
base1["cod_des"]=base1["cod_des"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxdesegrsop,how='left',on='cod_des')
base1=pd.merge(base1,cqxdesegrsop,how='inner',on='cod_des')
base1.shape

(41860, 25)

In [40]:
merge.shape

(41860, 25)

In [41]:
log_falla('id_desope', 'cod_des', True)
base1=base1.drop('cod_des',axis=1)
dim.append('dim_cqxdesegrsop')
control_d.append(base1.shape[0])

In [42]:
base1['sol_fec_temp'] = base1['sol_fec'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"sol_fec_temp"})
tiempo["sol_fec_temp"] = tiempo['sol_fec_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='sol_fec_temp')
base1 = pd.merge(base1, tiempo, how='left', on='sol_fec_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_sol","dt_ano":"id_ano_sol","dt_mes":"id_mes_sol",
                            "dt_dia":"id_dia_sol","dt_dia_sem":"id_diasem_sol","dt_sem":"id_sem_sol"})
base1=base1.drop("sol_fec_temp",axis=1)
base1.shape

(41860, 30)

In [43]:
tiempo

,id_tiempo,sol_fec_temp,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano
0,20130101,2013-01-01,1.0,1.0,2.0,1.0,2013.0
1,20130102,2013-01-02,1.0,2.0,3.0,1.0,2013.0
2,20130103,2013-01-03,1.0,3.0,4.0,1.0,2013.0
3,20130104,2013-01-04,1.0,4.0,5.0,1.0,2013.0
4,20130105,2013-01-05,1.0,5.0,6.0,1.0,2013.0
...,...,...,...,...,...,...,...
5474,20261227,2026-12-27,12.0,27.0,7.0,2.0,2026.0
5475,20261228,2026-12-28,12.0,28.0,1.0,2.0,2026.0
5476,20261229,2026-12-29,12.0,29.0,2.0,2.0,2026.0
5477,20261230,2026-12-30,12.0,30.0,3.0,2.0,2026.0


In [44]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 41860 entries, 0 to 41859
Data columns (total 30 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   sol_num        41860 non-null  float64            
 1   inf_ane        41860 non-null  float64            
 2   dur_ane        41860 non-null  object             
 3   fec_cre        41860 non-null  datetime64[ns, UTC]
 4   fec_mod        16064 non-null  datetime64[ns, UTC]
 5   dur_ane_ini    41860 non-null  object             
 6   dur_ane_fin    41860 non-null  object             
 7   dur_sal        41860 non-null  object             
 8   dur_sal_ini    41860 non-null  object             
 9   dur_sal_fin    41860 non-null  object             
 10  hor_ini_ope    41860 non-null  object             
 11  hor_fin_ope    41860 non-null  object             
 12  dur_ope        41860 non-null  object             
 13  sol_fec        41860 non-null  object         

In [45]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [46]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [47]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [48]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [49]:
base

,id_numdoc,dur_ope,dur_sal_fin,dur_ane_ini,fec_mod,id_usumod,id_red,id_diasem_sol,fec_cre,dur_ane,...,dur_sal_ini,sol_act,id_pacsec,dur_ane_fin,id_dia_sol,id_time_sol,sol_num,id_oricas,id_desope,id_mes_sol
0,7027283,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,NaT,NaN,29,1.0,2023-06-07 18:02:39+00:00,0001-01-01 00:00:00-04:56:16,...,0001-01-01 00:00:00-04:56:16,2045551.0,NaN,0001-01-01 00:00:00-04:56:16,8.0,20150608.0,2246.0,1,11,6.0
1,7027283,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,NaT,NaN,29,5.0,2023-06-13 21:09:48+00:00,0001-01-01 00:00:00-04:56:16,...,0001-01-01 00:00:00-04:56:16,2573331.0,54450881.0,0001-01-01 00:00:00-04:56:16,3.0,20220603.0,14925.0,1,11,6.0
2,6972137,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,2023-05-03 01:08:36+00:00,6972137.0,29,6.0,2023-05-02 22:23:33+00:00,0001-01-01 00:00:00-04:56:16,...,0001-01-01 00:00:00-04:56:16,2669333.0,54567599.0,0001-01-01 00:00:00-04:56:16,29.0,20230429.0,21632.0,1,11,4.0
3,6916651,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,NaT,NaN,29,5.0,2023-05-20 00:15:15+00:00,0001-01-01 00:00:00-04:56:16,...,0001-01-01 00:00:00-04:56:16,2661036.0,44751550.0,0001-01-01 00:00:00-04:56:16,19.0,20230519.0,21510.0,1,11,5.0
4,6969192,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,0001-01-01 00:00:00-04:56:16,NaT,NaN,29,2.0,2023-05-31 19:19:30+00:00,0001-01-01 00:00:00-04:56:16,...,0001-01-01 00:00:00-04:56:16,2662436.0,49878650.0,0001-01-01 00:00:00-04:56:16,30.0,20230530.0,22238.0,1,11,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41855,6965510,0001-01-01 02:10:00-04:56:16,0001-01-01 11:30:00-04:56:16,0001-01-01 09:00:00-04:56:16,NaT,NaN,20,5.0,2023-06-09 16:38:55+00:00,0001-01-01 02:30:00-04:56:16,...,0001-01-01 09:00:00-04:56:16,332518.0,53441130.0,0001-01-01 11:30:00-04:56:16,9.0,20230609.0,2265.0,1,1,6.0
41856,6965510,0001-01-01 00:15:00-04:56:16,0001-01-01 12:00:00-04:56:16,0001-01-01 11:30:00-04:56:16,NaT,NaN,20,4.0,2023-06-08 16:54:51+00:00,0001-01-01 00:30:00-04:56:16,...,0001-01-01 11:30:00-04:56:16,332058.0,42073711.0,0001-01-01 12:00:00-04:56:16,8.0,20230608.0,2261.0,1,1,6.0
41857,6965510,0001-01-01 01:10:00-04:56:16,0001-01-01 13:30:00-04:56:16,0001-01-01 12:00:00-04:56:16,NaT,NaN,20,2.0,2023-06-06 18:13:40+00:00,0001-01-01 01:30:00-04:56:16,...,0001-01-01 12:00:00-04:56:16,331337.0,56208705.0,0001-01-01 13:30:00-04:56:16,6.0,20230606.0,2256.0,1,1,6.0
41858,6965510,0001-01-01 02:10:00-04:56:16,0001-01-01 11:30:00-04:56:16,0001-01-01 09:00:00-04:56:16,NaT,NaN,20,4.0,2023-06-08 16:08:51+00:00,0001-01-01 02:30:00-04:56:16,...,0001-01-01 09:00:00-04:56:16,332136.0,50575207.0,0001-01-01 11:30:00-04:56:16,8.0,20230608.0,2259.0,1,1,6.0


In [50]:
base.columns

Index(['id_numdoc', 'dur_ope', 'dur_sal_fin', 'dur_ane_ini', 'fec_mod',
       'id_usumod', 'id_red', 'id_diasem_sol', 'fec_cre', 'dur_ane',
       'id_ano_sol', 'sol_fec', 'hor_fin_ope', 'dur_sal', 'id_tipdoc',
       'hor_ini_ope', 'inf_ane', 'id_usucre', 'id_sem_sol', 'id_cas',
       'dur_sal_ini', 'sol_act', 'id_pacsec', 'dur_ane_fin', 'id_dia_sol',
       'id_time_sol', 'sol_num', 'id_oricas', 'id_desope', 'id_mes_sol'],
      dtype='object')

In [51]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

4860

In [52]:
fecha_fin = "2024-12-31"

In [53]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [54]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_oricas,2023-05-01,2024-12-31,41860,41860,0,0,id_oricas
1,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_cas,2023-05-01,2024-12-31,41860,41860,0,0,id_cas
2,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_red,2023-05-01,2024-12-31,41860,41860,0,0,id_red
3,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_personal,2023-05-01,2024-12-31,41860,41860,0,0,id_usumod
4,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_personal,2023-05-01,2024-12-31,41860,41860,0,0,id_usucre
5,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_personal,2023-05-01,2024-12-31,41860,41860,0,0,id_numdoc
6,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_tipdoc,2023-05-01,2024-12-31,41860,41860,0,0,id_tipdoc
7,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_pacsec,2023-05-01,2024-12-31,41860,41860,0,0,id_pacsec
8,13,DAT,dat_ceqx003_essi,2023-06-26,15:50,15:52,dim_cqxdesegrsop,2023-05-01,2024-12-31,41860,41860,0,0,id_desope


In [55]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

9

In [56]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 54.673 segundos


In [57]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [58]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()